In [49]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

    !mkdir -p ~/.kaggle
    kaggle_config = {
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    }
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists("data"):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists("data"):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print("Data already exists")

Data already exists


In [50]:
import pandas as pd

df = pd.read_csv('data/train.csv')
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Fare', 'Cabin', 'Embarked'], axis=1)
df['Age'] = df['Age'].fillna(df['Age'].mean())


In [51]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

df['Sex'] = LabelEncoder().fit_transform(df['Sex'])

X = df.drop(['Survived'], axis=1)
Y = df['Survived']

X = MinMaxScaler().fit_transform(X)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, shuffle=True, stratify=Y)

In [52]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, KFold

model = SVC(kernel='rbf', coef0=0.3).fit(x_train, y_train)
p = model.predict(x_test)
score = accuracy_score(y_test, p)
cv = cross_val_score(model, x_train, y_train, cv=KFold(5))
print(cv.mean())

0.7865163006008077
